In [ ]:
# Import libraries
from delta import DeltaTable
from pyspark.sql.functions import format_string, when, col
import json

In [ ]:
# Define parameters with None, as values will come from pipeline activities
# Target
target_lakehouse_path = None 
target_schema = None
target_table = None

In [ ]:
# Define variables using values from parameters
target_path = f'{target_lakehouse_path}/Tables/{target_schema}/{target_table}'

# Load Helper Notebooks

Load helper notebook with the schema, merge keys and indicator mapping definition of gold tables inside "gold_objects" dictionary: 


In [ ]:
%run ./NB_Gold_Table_Schemas

In [ ]:
# Use gold_objects dictionary to define aditional variables
target_table_schema = gold_objects[f"{target_table}"]["schema"]

# Read and Transform New Data

In [ ]:
# Generate one row per hour 
df = spark.range(24).select(
    col("id").cast("int").alias("hour_key")
)

# Add descriptive attributes
df = (
    df
    .withColumn(
        "hour_label",
        format_string("%02d:00", col("hour_key"))
    )
    .withColumn(
        "hour_range",
        format_string(
            "%02d:00-%02d:00",
            col("hour_key"),
            col("hour_key") + 1
        )
    )
    .withColumn(
        "day_period",
        when(col("hour_key") < 6, "Early Morning")
         .when(col("hour_key") < 12, "Morning")
         .when(col("hour_key") < 15, "Midday")
         .when(col("hour_key") < 20, "Afternoon")
         .otherwise("Night")
    )
    .withColumn(
        "day_period_key",
        when(col("hour_key") < 6, 1)
         .when(col("hour_key") < 12, 2)
         .when(col("hour_key") < 15, 3)
         .when(col("hour_key") < 20, 4)
         .otherwise(5)
    )
)

In [ ]:
# Order by hour_key
df = df.orderBy('hour_key')

# Update Table

In [ ]:
# Full load
# Create the table if it doesnt exists
try:
    delta_table = DeltaTable.forPath(spark, target_path)
    print('Table already exists')
except:
    delta_table = DeltaTable.createIfNotExists(spark) \
    .location(target_path) \
    .addColumns(target_table_schema) \
    .execute()
    print('Table has been created')

# Recreate the table without infering schema
df.write.format("delta").mode("overwrite").save(target_path) 

# Statistics
rows_inserted = df.count()
rows_updated = 0

In [ ]:
# File a dictionary with operational statistics
result = {
    "maxdate": None,
    "rows_inserted": rows_inserted,
    "rows_updated": rows_updated
}

# Make the result values queriable for other activities in the pipeline
mssparkutils.notebook.exit(json.dumps(result))